# 02 — Sullivan (2024) binary data retrieval, preprocessing, and star–planet mapping

**Purpose:** retrieve the Sullivan et al. (2024) catalogue, save its source tables, map each planet row to its binary-system stellar properties using the system-level `KOI`, and save the mapped dataset.

**Catalogue:** VizieR `J/AJ/168/129` (Sullivan et al. 2024) only.

**Important:** `KOI` identifies the host system; `KOIpl` identifies an individual planet. The merge is therefore many planets to one stellar system. It preserves all planet rows and does not apply the later MIST quality cut. Notebook 04 will create the MIST-quality-controlled planet-level table.

**Run from the project root.**

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from astroquery.vizier import Vizier

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw" / "sullivan"
PROCESSED_DIR = ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT.resolve())
print("Raw Sullivan directory:", RAW_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())

Project root: C:\Users\Lenovo\Downloads\Binaries
Raw Sullivan directory: C:\Users\Lenovo\Downloads\Binaries\data\raw\sullivan
Processed directory: C:\Users\Lenovo\Downloads\Binaries\data\processed


## 1. Retrieve and save the original Sullivan 2024 tables

The retrieval uses the catalogue identifier, rather than a hand-coded URL or a later paper's table. Raw tables are saved unchanged so the mapping can be reproduced.

In [2]:
Vizier.ROW_LIMIT = -1
CATALOGUE_ID = "J/AJ/168/129"

print(f"Retrieving VizieR catalogue {CATALOGUE_ID} ...")
tables = Vizier.get_catalogs(CATALOGUE_ID)
print("Tables returned:", len(tables))

if len(tables) != 4:
    raise RuntimeError(
        f"Expected 4 tables from {CATALOGUE_ID}; received {len(tables)}. "
        "Check the VizieR table structure before proceeding."
    )

sample_df = tables[0].to_pandas()
auxiliary_df = tables[1].to_pandas()
planet_df = tables[2].to_pandas()
notes_df = tables[3].to_pandas()

source_tables = {
    "sullivan_sample_table.csv": sample_df,
    "sullivan_auxiliary_table.csv": auxiliary_df,
    "sullivan_planet_table.csv": planet_df,
    "sullivan_notes_table.csv": notes_df,
}
for filename, df in source_tables.items():
    path = RAW_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved {path.relative_to(ROOT)} | shape={df.shape}")

Retrieving VizieR catalogue J/AJ/168/129 ...
Tables returned: 4
Saved data\raw\sullivan\sullivan_sample_table.csv | shape=(286, 52)
Saved data\raw\sullivan\sullivan_auxiliary_table.csv | shape=(82, 13)
Saved data\raw\sullivan\sullivan_planet_table.csv | shape=(404, 28)
Saved data\raw\sullivan\sullivan_notes_table.csv | shape=(65, 8)


## 2. Validate system and planet identifiers

`KOI` must be an integer-valued system identifier. The normalizer handles numeric values and strings with leading zeros, but raises an error if a supposed system ID has a non-integer part. We check that the stellar table has one row per system and the planet table has one row per `KOIpl`.

In [3]:
def normalize_system_koi(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    fractional = numeric.notna() & (~np.isclose(numeric, np.round(numeric), atol=1e-8, rtol=0.0))
    if fractional.any():
        examples = series.loc[fractional].head(10).tolist()
        raise ValueError(f"Non-integer values found in system KOI column: {examples}")
    return numeric.round().astype("Int64")

# Reload the exact files saved above to ensure subsequent steps use persisted inputs.
stars = pd.read_csv(RAW_DIR / "sullivan_sample_table.csv", dtype={"KOI": "string"})
planets = pd.read_csv(RAW_DIR / "sullivan_planet_table.csv", dtype={"KOI": "string", "KOIpl": "string"})

for label, df, required_cols in [
    ("stellar sample", stars, ["KOI"]),
    ("planet table", planets, ["KOI", "KOIpl"]),
]:
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns in {label}: {sorted(missing)}")

stars["KOI_key"] = normalize_system_koi(stars["KOI"])
planets["KOI_key"] = normalize_system_koi(planets["KOI"])

print("STELLAR SAMPLE")
print("Rows:", len(stars))
print("Unique systems:", stars["KOI_key"].nunique(dropna=True))
print("Missing system IDs:", stars["KOI_key"].isna().sum())
print("Duplicate system IDs:", stars["KOI_key"].duplicated().sum())

print("\nPLANET TABLE")
print("Planet rows:", len(planets))
print("Unique KOIpl identifiers:", planets["KOIpl"].nunique(dropna=True))
print("Missing system IDs:", planets["KOI_key"].isna().sum())
print("Missing KOIpl identifiers:", planets["KOIpl"].isna().sum())
print("Duplicate KOIpl identifiers:", planets["KOIpl"].duplicated().sum())

if stars["KOI_key"].isna().any():
    raise ValueError("The stellar table contains missing system KOI identifiers.")
if stars["KOI_key"].duplicated().any():
    raise ValueError("The stellar table has duplicate system KOIs; a many-to-one merge is unsafe.")
if planets["KOIpl"].duplicated().any():
    raise ValueError("The planet table has duplicate KOIpl identifiers; inspect before mapping.")

STELLAR SAMPLE
Rows: 286
Unique systems: 286
Missing system IDs: 0
Duplicate system IDs: 0

PLANET TABLE
Planet rows: 404
Unique KOIpl identifiers: 404
Missing system IDs: 0
Missing KOIpl identifiers: 0
Duplicate KOIpl identifiers: 0


## 3. Map each planet to its binary-system stellar properties

This is a left join from the planet table onto the stellar-system table. Thus every original planet row is retained, including rows whose host-system match is missing. Both original system identifiers are preserved for audit; `KOI` in the output is the normalized system key, and `KOIpl` remains the planet-level identifier.

In [4]:
# Preserve source identifiers before joining.
stars_for_merge = stars.copy().rename(columns={"KOI": "KOI_star_catalogue"})
planets_for_merge = planets.copy().rename(columns={"KOI": "KOI_planet_catalogue"})

mapped_all = planets_for_merge.merge(
    stars_for_merge,
    on="KOI_key",
    how="left",
    validate="many_to_one",
    indicator="_stellar_match",
    suffixes=("_planet", "_star"),
)

mapped_all.insert(0, "KOI", mapped_all["KOI_key"].astype("Int64"))
mapped_all["stellar_match"] = mapped_all["_stellar_match"].eq("both")
mapped_all = mapped_all.drop(columns=["_stellar_match"])

mapped_matched = mapped_all.loc[mapped_all["stellar_match"]].copy()
mapped_unmatched = mapped_all.loc[~mapped_all["stellar_match"]].copy()

print("Original planet rows:", len(planets))
print("Mapped planet rows with stellar match:", len(mapped_matched))
print("Planet rows without stellar match:", len(mapped_unmatched))
print("Unique matched systems:", mapped_matched["KOI"].nunique())
print("Unique planet identifiers in output:", mapped_all["KOIpl"].nunique())

# Show a compact mapping check.
show_cols = [c for c in [
    "KOI", "KOI_planet_catalogue", "KOI_star_catalogue", "KOIpl",
    "Sep", "Teff1", "Teff2", "Rstar1", "Rstar2/Rstar1",
    "Rpkep", "Rppri", "Rpsec", "stellar_match",
] if c in mapped_all.columns]
display(mapped_all[show_cols].head(12))

Original planet rows: 404
Mapped planet rows with stellar match: 404
Planet rows without stellar match: 0
Unique matched systems: 286
Unique planet identifiers in output: 404


,KOI,KOI_planet_catalogue,KOI_star_catalogue,KOIpl,Sep,Teff1,Teff2,Rstar1,Rstar2/Rstar1,Rpkep,Rppri,Rpsec,stellar_match
0,42,0042,0042,42.01,1.66,6640,4771,1.376,0.516,2.43,2.28,4.89,True
1,112,0112,0112,112.01,0.11,6256,5506,1.124,0.753,2.75,3.55,4.74,True
2,112,0112,0112,112.02,0.11,6256,5506,1.124,0.753,1.16,1.49,1.98,True
3,162,0162,0162,162.01,0.29,6148,5622,1.078,0.811,3.11,3.72,4.50,True
4,163,0163,0163,163.01,1.22,5290,5015,0.763,0.956,2.08,2.71,3.09,True
5,165,0165,0165,165.01,0.28,5508,4432,0.850,0.772,2.38,2.79,4.93,True
6,174,0174,0174,174.01,0.56,4954,3508,0.728,0.450,2.59,2.40,7.15,True
7,177,0177,0177,177.01,0.23,6085,5903,1.051,0.934,1.78,2.45,2.62,True
8,191,0191,0191,191.01,1.67,5685,3964,0.882,0.604,10.89,10.64,29.08,True
9,191,0191,0191,191.02,1.67,5685,3964,0.882,0.604,2.25,2.20,6.01,True


## 4. Save mapped data and an audit summary

The primary output contains all Sullivan planet rows and mapped stellar columns; a separate unmatched file makes any failed match explicit. Notebook 04 will use the MIST-quality-controlled stellar sample to produce the final analysis table.

In [5]:
MAPPED_PATH = PROCESSED_DIR / "sullivan2024_planets_with_stellar_properties.csv"
MATCHED_PATH = PROCESSED_DIR / "sullivan2024_planets_with_stellar_match.csv"
UNMATCHED_PATH = PROCESSED_DIR / "sullivan2024_planets_without_stellar_match.csv"
AUDIT_PATH = PROCESSED_DIR / "sullivan2024_star_planet_mapping_audit.csv"

mapped_all.to_csv(MAPPED_PATH, index=False)
mapped_matched.to_csv(MATCHED_PATH, index=False)
mapped_unmatched.to_csv(UNMATCHED_PATH, index=False)

# Number of planets per system is a useful many-planets-to-one-star check.
planets_per_system = mapped_matched.groupby("KOI", dropna=True)["KOIpl"].size()
audit = pd.DataFrame([
    {"metric": "source_stellar_system_rows", "value": len(stars)},
    {"metric": "source_unique_stellar_systems", "value": stars["KOI_key"].nunique()},
    {"metric": "source_planet_rows", "value": len(planets)},
    {"metric": "source_unique_KOIpl", "value": planets["KOIpl"].nunique()},
    {"metric": "mapped_planet_rows_all", "value": len(mapped_all)},
    {"metric": "planet_rows_with_stellar_match", "value": len(mapped_matched)},
    {"metric": "planet_rows_without_stellar_match", "value": len(mapped_unmatched)},
    {"metric": "unique_matched_systems", "value": mapped_matched["KOI"].nunique()},
    {"metric": "systems_with_multiple_planets", "value": int((planets_per_system > 1).sum())},
    {"metric": "maximum_planets_per_matched_system", "value": int(planets_per_system.max()) if len(planets_per_system) else 0},
])
audit.to_csv(AUDIT_PATH, index=False)

print("Saved mapped data:", MAPPED_PATH)
print("Saved matched-only data:", MATCHED_PATH)
print("Saved unmatched rows:", UNMATCHED_PATH)
print("Saved audit:", AUDIT_PATH)
display(audit)

# Final integrity assertions.
assert len(mapped_all) == len(planets), "Left join unexpectedly changed the planet row count."
assert mapped_all["KOIpl"].nunique() == planets["KOIpl"].nunique(), "Some planet identifiers were lost or duplicated."
print("\nMapping integrity checks passed.")

Saved mapped data: C:\Users\Lenovo\Downloads\Binaries\data\processed\sullivan2024_planets_with_stellar_properties.csv
Saved matched-only data: C:\Users\Lenovo\Downloads\Binaries\data\processed\sullivan2024_planets_with_stellar_match.csv
Saved unmatched rows: C:\Users\Lenovo\Downloads\Binaries\data\processed\sullivan2024_planets_without_stellar_match.csv
Saved audit: C:\Users\Lenovo\Downloads\Binaries\data\processed\sullivan2024_star_planet_mapping_audit.csv


,metric,value
0,source_stellar_system_rows,286
1,source_unique_stellar_systems,286
2,source_planet_rows,404
3,source_unique_KOIpl,404
4,mapped_planet_rows_all,404
5,planet_rows_with_stellar_match,404
6,planet_rows_without_stellar_match,0
7,unique_matched_systems,286
8,systems_with_multiple_planets,77
9,maximum_planets_per_matched_system,5



Mapping integrity checks passed.
